In [1]:
#| default_exp core.attention
#| export

In [2]:
#| export

import numpy as np
import math
import time
from typing import Optional, Tuple, List

# Import dependencies from previous modules - following TinyTorch dependency chain
from tinytorch.core.tensor import Tensor
from tinytorch.core.layers import Linear
from tinytorch.core.activations import Softmax

# Constants for attention computation
MASK_VALUE = -1e9  # Large negative value used for attention masking (becomes ~0 after softmax)

In [3]:
#| export
def _compute_attention_scores(Q: Tensor, K: Tensor) -> Tensor:
    """Compute raw attention scores via Q @ K^T.

    TODO: Transpose K and multiply by Q to get similarity matrix

    APPROACH:
    1. Transpose K: swap last two dims so (batch, seq, d) -> (batch, d, seq)
    2. Matrix multiply: Q @ K^T gives (batch, seq, seq) scores

    EXAMPLE:
    >>> Q = Tensor(np.random.randn(1, 3, 4))  # 3 tokens, dim=4
    >>> K = Tensor(np.random.randn(1, 3, 4))
    >>> scores = _compute_attention_scores(Q, K)
    >>> print(scores.shape)  # (1, 3, 3) -- every token scored against every other

    HINT: Use K.transpose(-2, -1) to swap the last two dimensions
    """
    ### BEGIN SOLUTION
    K_t = K.transpose(-2, -1)
    return Q.matmul(K_t)
    ### END SOLUTION

In [6]:
def test_unit_attention_scores():
    """🧪 Test attention score computation."""
    print("🧪 Unit Test: Attention Scores...")
    Q = Tensor(np.ones((1, 3, 4)))
    print(Q)
    K = Tensor(np.ones((1, 3, 4)))
    print(K)
    scores = _compute_attention_scores(Q, K)
    print(scores)
    assert scores.shape == (1, 3, 3), f"Expected (1,3,3), got {scores.shape}"
    assert np.allclose(scores.data, 4.0), "All-ones Q@K^T should give d_model=4"
    print("✅ Attention scores: correct shapes and values!")

if __name__ == "__main__":
    test_unit_attention_scores()

🧪 Unit Test: Attention Scores...
Tensor([[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]])
Tensor([[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]])
Tensor([[[4. 4. 4.]
  [4. 4. 4.]
  [4. 4. 4.]]])
✅ Attention scores: correct shapes and values!


In [7]:
#| export
def _scale_scores(scores: Tensor, d_model: int) -> Tensor:
    """Scale attention scores by 1/sqrt(d_model).

    TODO: Divide scores by the square root of the model dimension

    APPROACH:
    1. Compute scale factor: 1.0 / math.sqrt(d_model)
    2. Multiply scores by scale factor

    EXAMPLE:
    >>> scores = Tensor(np.array([[[4.0, 8.0]]]))
    >>> scaled = _scale_scores(scores, d_model=4)
    >>> print(scaled.data)  # [[[ 2.0, 4.0]]] -- divided by sqrt(4)=2

    HINT: Use math.sqrt() for the square root
    """
    ### BEGIN SOLUTION
    scale_factor = 1.0 / math.sqrt(d_model)
    return scores * scale_factor
    ### END SOLUTION

In [12]:
def test_unit_scale_scores():
    """🧪 Test attention score scaling."""
    print("🧪 Unit Test: Score Scaling...")
    scores = Tensor(np.array([[[4.0, 8.0]]]))
    scaled = _scale_scores(scores, d_model=4)
    # print(scaled)
    assert np.allclose(scaled.data, [[[2.0, 4.0]]]), f"Expected /sqrt(4)=2, got {scaled.data}"
    print("✅ Score scaling works correctly!")

if __name__ == "__main__":
    test_unit_scale_scores()

🧪 Unit Test: Score Scaling...
✅ Score scaling works correctly!


In [15]:
#| export
def _apply_mask(scores: Tensor, mask: Tensor) -> Tensor:
    """Apply causal mask by setting masked positions to -infinity.

    TODO: Add large negative values to positions where mask is 0

    APPROACH:
    1. Compute additive mask: (1 - mask) * MASK_VALUE
    2. Add to scores (masked positions become -inf, unmasked unchanged)

    EXAMPLE:
    >>> scores = Tensor(np.ones((1, 3, 3)))
    >>> mask = Tensor(np.tril(np.ones((1, 3, 3))))  # lower triangle
    >>> masked = _apply_mask(scores, mask)
    >>> print(masked.data[0, 0, 1])  # -1e9 (future position masked)

    HINT: mask=0 means "block this position", mask=1 means "allow"
    """
    ### BEGIN SOLUTION
    adder = (1.0 - mask.data) * MASK_VALUE
    return scores + Tensor(adder)
    ### END SOLUTION

In [21]:
def test_unit_apply_mask():
    """🧪 Test causal mask application."""
    print("🧪 Unit Test: Causal Masking...")
    scores = Tensor(np.ones((1, 3, 3)))
    mask = Tensor(np.tril(np.ones((1, 3, 3))))
    # print(mask)
    masked = _apply_mask(scores, mask)
    # print(masked)
    # Future positions should be large negative
    assert masked.data[0, 0, 1] < -1e8, "Future position not masked"
    # Past positions should be unchanged
    assert np.allclose(masked.data[0, 0, 0], 1.0), "Past position was modified"
    print("✅ Causal masking works correctly!")

if __name__ == "__main__":
    test_unit_apply_mask()

🧪 Unit Test: Causal Masking...
✅ Causal masking works correctly!


In [22]:
#| export
def scaled_dot_product_attention(Q: Tensor, K: Tensor, V: Tensor, mask: Optional[Tensor] = None) -> Tuple[Tensor, Tensor]:
    """Complete scaled dot-product attention.

    TODO: Compose the helpers into the full attention operation

    APPROACH:
    1. Call _compute_attention_scores(Q, K) for raw similarity
    2. Call _scale_scores(scores, Q.shape[-1]) for numerical stability
    3. If mask provided, call _apply_mask(scores, mask)
    4. Apply Softmax to get probability weights
    5. Multiply weights @ V for attended values

    SUB-PROBLEMS (you already implemented these):
    - _compute_attention_scores: Q @ K^T similarity matrix
    - _scale_scores: divide by sqrt(d) for stable softmax
    - _apply_mask: block future positions with -inf

    Args:
        Q: Query tensor of shape (batch_size, seq_len, d_model)
        K: Key tensor of shape (batch_size, seq_len, d_model)
        V: Value tensor of shape (batch_size, seq_len, d_model)
        mask: Optional causal mask, True=allow, False=mask (batch_size, seq_len, seq_len)

    Returns:
        output: Attended values (batch_size, seq_len, d_model)
        attention_weights: Attention matrix (batch_size, seq_len, seq_len)

    EXAMPLE:
    >>> Q = Tensor(np.random.randn(2, 4, 64))
    >>> K = Tensor(np.random.randn(2, 4, 64))
    >>> V = Tensor(np.random.randn(2, 4, 64))
    >>> output, weights = scaled_dot_product_attention(Q, K, V)
    >>> print(output.shape)   # (2, 4, 64)
    >>> print(weights.shape)  # (2, 4, 4)

    HINT: Softmax is already imported -- use Softmax()(scores, dim=-1)
    """
    ### BEGIN SOLUTION
    scores = _compute_attention_scores(Q, K)
    scores = _scale_scores(scores, Q.shape[-1])
    if mask is not None:
        scores = _apply_mask(scores, mask)
    softmax = Softmax()
    attention_weights = softmax(scores, dim=-1)
    output = attention_weights.matmul(V)
    return output, attention_weights
    ### END SOLUTION

In [23]:
def test_unit_scaled_dot_product_attention():
    """🧪 Test scaled dot-product attention implementation."""
    print("🧪 Unit Test: Scaled Dot-Product Attention...")

    # Test basic functionality
    batch_size, seq_len, d_model = 2, 4, 8
    Q = Tensor(np.random.randn(batch_size, seq_len, d_model))
    K = Tensor(np.random.randn(batch_size, seq_len, d_model))
    V = Tensor(np.random.randn(batch_size, seq_len, d_model))

    output, weights = scaled_dot_product_attention(Q, K, V)

    # Check output shapes
    assert output.shape == (batch_size, seq_len, d_model), f"Output shape {output.shape} incorrect"
    assert weights.shape == (batch_size, seq_len, seq_len), f"Weights shape {weights.shape} incorrect"

    # Check attention weights sum to 1 (probability distribution)
    weights_sum = weights.data.sum(axis=2)  # Sum over last dimension
    expected_sum = np.ones((batch_size, seq_len))
    assert np.allclose(weights_sum, expected_sum, atol=1e-6), "Attention weights don't sum to 1"

    # Test with causal mask
    mask = Tensor(np.tril(np.ones((batch_size, seq_len, seq_len)), k=0))  # Lower triangular
    output_masked, weights_masked = scaled_dot_product_attention(Q, K, V, mask)

    # Check that future positions have zero attention
    for b in range(batch_size):
        for i in range(seq_len):
            for j in range(i + 1, seq_len):  # Future positions
                assert abs(weights_masked.data[b, i, j]) < 1e-6, f"Future attention not masked at ({i},{j})"

    print("✅ scaled_dot_product_attention works correctly!")

if __name__ == "__main__":
    test_unit_scaled_dot_product_attention()

🧪 Unit Test: Scaled Dot-Product Attention...
✅ scaled_dot_product_attention works correctly!
